<a href="https://colab.research.google.com/github/3702880330101-netizen/Smart-Cost/blob/main/SmartCost.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# Instalar la librería oficial de Gemini si no está instalada
!pip install -q google-generativeai

import requests
import pandas as pd
import numpy as np
import google.generativeai as genai
from google.colab import userdata

# ==========================================
# 0. CONFIGURACIÓN SEGURA DE LA API DE GEMINI
# ==========================================
try:
    API_KEY = userdata.get('GEMINI_API_KEY')
    genai.configure(api_key=API_KEY)
    print("¡Clave de API configurada de forma segura desde los Secretos de Colab!")
except Exception as e:
    print("Error: Asegúrate de guardar tu clave en la sección de Secretos (icono de llave en el panel izquierdo de Colab) con el nombre exacto 'GEMINI_API_KEY'. Detalle:", e)

# ==========================================
# 1. CARGA AUTOMÁTICA DE DATOS DESDE GITHUB/GIST
# ==========================================
url_json = "https://gist.githubusercontent.com/3702880330101-netizen/3423b40e68604e988e434002708f03d9/raw/costos_data.json"

try:
    response = requests.get(url_json)
    response.raise_for_status()
    data = response.json()
    print("¡Datos cargados exitosamente desde tu Gist en GitHub!")
except Exception as e:
    print("Aviso: No se pudo conectar al Gist, usando valores de respaldo por defecto:", e)
    data = {
        "material": "Aluminio",
        "consumo_estandar": 1.0,
        "precio_material": 50.00,
        "desperdicio": 5.0,
        "horas_mod": 1.0,
        "tarifa_mod": 250.00,
        "cif_variable": 250.00,
        "costos_fijos": 400.00,
        "volumen": 200,
        "precio_venta": 250.00
    }

# Mostrar los datos base en formato de tabla limpia
df_base = pd.DataFrame([data])
print("\n--- DATOS ACTUALES RECIBIDOS DE LA WEB ---")
display(df_base)


# ==========================================
# 2. MOTOR DE CÁLCULO Y SENSIBILIDAD
# ==========================================
def calcular_modelo(p):
    material_ajustado = p['consumo_estandar'] * (1 + (p['desperdicio'] / 100))
    costo_mp_unit = material_ajustado * p['precio_material']
    costo_mod_unit = p['horas_mod'] * p['tarifa_mod']
    costo_var_unit = costo_mp_unit + costo_mod_unit + p['cif_variable']
    costo_total = (costo_var_unit * p['volumen']) + p['costos_fijos']
    costo_unitario = costo_total / p['volumen'] if p['volumen'] > 0 else 0
    margen_contrib_unit = p['precio_venta'] - costo_var_unit
    punto_equilibrio = np.ceil(p['costos_fijos'] / margen_contrib_unit) if margen_contrib_unit > 0 else 0
    utilidad = (margen_contrib_unit * p['volumen']) - p['costos_fijos']

    return {
        "Costo Unitario": costo_unitario,
        "Costo Variable Unitario": costo_var_unit,
        "Margen de Contribución": margen_contrib_unit,
        "Punto de Equilibrio (u.)": punto_equilibrio,
        "Utilidad Total": utilidad
    }

res_base = calcular_modelo(data)
print("\n--- RESULTADOS DEL ESCENARIO BASE ---")
for k, v in res_base.items():
    print(f"• {k}: Q{v:,.2f}" if isinstance(v, float) else f"• {k}: {v}")


# ==========================================
# 3. PRIORIZACIÓN DE VARIABLES Y ESCENARIOS
# ==========================================
variables_a_probar = ['precio_venta', 'precio_material', 'volumen', 'desperdicio', 'horas_mod', 'costos_fijos']
impactos = []
utilidad_base = res_base['Utilidad Total']

for var in variables_a_probar:
    data_mod = data.copy()
    data_mod[var] = data_mod[var] * 1.10
    res_mod = calcular_modelo(data_mod)
    dif_utilidad = res_mod['Utilidad Total'] - utilidad_base
    impactos.append({
        "Variable": var,
        "Impacto en Utilidad por +10% (Q)": dif_utilidad
    })

df_impactos = pd.DataFrame(impactos).sort_values(by="Impacto en Utilidad por +10% (Q)", ascending=False)
print("\n--- PRIORIZACIÓN DE VARIABLES (Impacto en Utilidad) ---")
display(df_impactos)

escenarios = {
    "Base Actual": data.copy(),
    "Reducción de Desperdicio (2%)": {**data, "desperdicio": 2.0},
    "Optimización de MP (-5% Precio)": {**data, "precio_material": data["precio_material"] * 0.95},
    "Aumento de Volumen (+5%)": {**data, "volumen": int(data["volumen"] * 1.05)}
}

resultados_escenarios = []
for nombre, params in escenarios.items():
    res = calcular_modelo(params)
    resultados_escenarios.append({
        "Escenario": nombre,
        "Costo Unitario (Q)": res["Costo Unitario"],
        "Punto de Equilibrio": res["Punto de Equilibrio (u.)"],
        "Utilidad Total (Q)": res["Utilidad Total"]
    })

df_escenarios = pd.DataFrame(resultados_escenarios)
print("\n--- COMPARATIVA DE ESCENARIOS DE MEJORA ---")
display(df_escenarios)


# ==========================================
# 4. REPORTE GERENCIAL AUTOMATIZADO CON GEMINI
# ==========================================
prompt = f"""
Actúa como un Director Financiero y Consultor experto en Costos Industriales.
Analiza los siguientes resultados numéricos basados en Quetzales (Q) y redacta un informe gerencial ejecutivo y profesional:

- Datos de entrada del producto: {data}
- Resultados Financieros Base: {res_base}
- Priorización de Impacto por Variable (+10%):
{df_impactos.to_string(index=False)}
- Comparativa de Escenarios de Mejora:
{df_escenarios.to_string(index=False)}

El informe debe estructurarse con claridad profesional e incluir:
1. Resumen ejecutivo de la rentabilidad actual.
2. Análisis de sensibilidad (cuál variable afecta más críticamente la utilidad).
3. Evaluación del punto de equilibrio operativo.
4. Conclusiones y recomendaciones estratégicas basadas en los escenarios simulados.
"""

try:
    model = genai.GenerativeModel("gemini-1.5-flash")
    response = model.generate_content(prompt)

    print("\n" + "=".center(60, "="))
    print(" REPORTE EJECUTIVO GERENCIAL GENERADO POR IA (GEMINI) ".center(60, "="))
    print("=".center(60, "="))
    print(response.text)
    print("="*60)
except Exception as e:
    print("Aviso: No se pudo generar el reporte con la API de Gemini:", e)

¡Clave de API configurada de forma segura desde los Secretos de Colab!
¡Datos cargados exitosamente desde tu Gist en GitHub!

--- DATOS ACTUALES RECIBIDOS DE LA WEB ---


,material,consumo_estandar,precio_material,desperdicio,horas_mod,tarifa_mod,cif_variable,costos_fijos,volumen,precio_venta,timestamp
0,Acero Inoxidable,1.0,50.0,5.0,2.5,15.0,10.0,5000.0,1000,120.0,2026-10-07T01:42:00.000Z



--- RESULTADOS DEL ESCENARIO BASE ---
• Costo Unitario: Q105.00
• Costo Variable Unitario: Q100.00
• Margen de Contribución: Q20.00
• Punto de Equilibrio (u.): Q250.00
• Utilidad Total: Q15,000.00

--- PRIORIZACIÓN DE VARIABLES (Impacto en Utilidad) ---


,Variable,Impacto en Utilidad por +10% (Q)
0,precio_venta,12000.0
2,volumen,2000.0
3,desperdicio,-250.0
5,costos_fijos,-500.0
4,horas_mod,-3750.0
1,precio_material,-5250.0



--- COMPARATIVA DE ESCENARIOS DE MEJORA ---


,Escenario,Costo Unitario (Q),Punto de Equilibrio,Utilidad Total (Q)
0,Base Actual,105.000000,250.0,15000.0
1,Reducción de Desperdicio (2%),103.500000,233.0,16500.0
2,Optimización de MP (-5% Precio),102.375000,221.0,17625.0
3,Aumento de Volumen (+5%),104.761905,250.0,16000.0


Aviso: No se pudo generar el reporte con la API de Gemini: 401 API key not valid. Please pass a valid API key. [reason: "ACCESS_TOKEN_TYPE_UNSUPPORTED"
metadata {
  key: "service"
  value: "generativelanguage.googleapis.com"
}
metadata {
  key: "method"
  value: "google.ai.generativelanguage.v1beta.GenerativeService.GenerateContent"
}
]
